In [1]:
import imaplib
import email
from email.header import decode_header
import re
import pandas as pd

In [2]:
# Configuración
EMAIL = "diego.vaca.enriquez@gmail.com"
PASSWORD = "djol xhos knrg emhd"

# Palabras clave típicas de consumos
KEYWORDS = "Notificación de consumos"

In [3]:
# Conexión a Gmail
mail = imaplib.IMAP4_SSL("imap.gmail.com")
mail.login(EMAIL, PASSWORD)
mail.select("INBOX")

('OK', [b'38914'])

In [5]:
# Buscar todos los correos
status, messages = mail.search(None, 'SINCE "17-Jul-2026"')

resultados = []

for num in messages[0].split():
    status, data = mail.fetch(num, "(RFC822)")

    for response in data:
        if not isinstance(response, tuple):
            continue

        msg = email.message_from_bytes(response[1])

        asunto = msg.get("Subject", "")

        try:
            decoded = decode_header(asunto)[0]
            if isinstance(decoded[0], bytes):
                asunto = decoded[0].decode(
                    decoded[1] if decoded[1] else "utf-8",
                    errors="ignore"
                )
        except:
            pass

        remitente = msg.get("From", "")
        fecha = msg.get("Date", "")

        cuerpo = ""

        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    try:
                        cuerpo += part.get_payload(decode=True).decode(
                            errors="ignore"
                        )
                    except:
                        pass
        else:
            try:
                cuerpo = msg.get_payload(decode=True).decode(errors="ignore")
            except:
                pass

        texto = f"{asunto}\n{cuerpo}".lower()

        if any(texto for k in KEYWORDS):
            resultados.append({
                "fecha": fecha,
                "remitente": remitente,
                "asunto": asunto
            })

df = pd.DataFrame(resultados)

df.to_excel("output/consumos_tarjeta.xlsx", index=False)

display(df)
print(f"Encontrados {len(df)} correos relacionados.")

,fecha,remitente,asunto
0,"Fri, 17 Jul 2026 11:02:00 +0000",=?UTF-8?B?QWlyIEV1cm9wYQ==?= <info@aireuropane...,🔜 Quedan pocos días para su vuelo a Frankfurt ...
1,"Fri, 17 Jul 2026 06:48:20 -0500 (GMT-05:00)",serviciosdigitales@dinersclub.com.ec,¡Hola! Ingresaste a blu.
2,"Fri, 17 Jul 2026 13:15:18 +0000","""Hume Health"" <support@myhumehealth.com>",Reminder: Your Store Credit Is Waiting.
3,"Fri, 17 Jul 2026 09:36:52 -0500 (GMT-05:00)",servicios@titanium.com.ec,Notificación de Consumos
4,"Fri, 17 Jul 2026 09:50:17 -0500 (GMT-05:00)",servicios@titanium.com.ec,Notificación de Consumos
...,...,...,...
273,"Wed, 05 Aug 2026 07:32:54 -0700","""LinkedIn"" <linkedin@em.linkedin.com>",Nuevos empleos anunciados.
274,"Wed, 05 Aug 2026 12:44:22 -0700",Steam <noreply@steampowered.com>,"¡Lies of P, de tu lista de deseados de Steam, ..."
275,"Wed, 05 Aug 2026 13:16:24 -0700",Google Assistant <googleassistant-noreply@goog...,El Asistente de Google dejará de estar disponi...
276,"Wed, 05 Aug 2026 20:46:32 +0000 (UTC)",logininfo@pichincha.com,NOTIFICACION BANCO PICHINCHA


Encontrados 278 correos relacionados.
